# Temporal Point Process Data Preprocessing

In [ ]:
import os
import sys
import json
import pickle
import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

In [2]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [3]:
data_folder = os.path.join('..', 'data')

## US Earthquakes

### Downloading Data

Download the US earthquake data from 2020-01-01 (inclusive) to 2024-01-01 (exclusive) to `data/raw/us_earthquake`.

In [4]:
from io import StringIO
from datetime import datetime, timedelta

In [5]:
def download_earthquake_data_chunk(start_time, end_time, region):
    """Download earthquake data for a given time chunk."""
    
    # USGS Earthquake API endpoint
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    
    # Query parameters
    params = {
        "format": "csv",           # Output format
        "starttime": start_time,    # Start date (YYYY-MM-DD)
        "endtime": end_time,        # End date (YYYY-MM-DD)
        # "minmagnitude": min_magnitude,  # Minimum magnitude
        # "maxmagnitude": max_magnitude,  # Maximum magnitude
        "minlatitude": region["minlatitude"],  # Min latitude of region
        "maxlatitude": region["maxlatitude"],  # Max latitude
        "minlongitude": region["minlongitude"],  # Min longitude of region
        "maxlongitude": region["maxlongitude"],  # Max longitude of region
    }
    
    # Send the request
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        print(f"Data downloaded successfully for {start_time} to {end_time}.")
        return response.content
    else:
        print(f"Failed to download data for {start_time} to {end_time}. HTTP Status Code: {response.status_code}.")
        return None

In [6]:
def download_earthquake_data(start_time, end_time, region, output_file, chunk_size):
    """Download earthquake data by splitting the request into smaller chunks."""
    
    # Convert start and end times to datetime objects
    start_date = datetime.strptime(start_time, "%Y-%m-%d")
    end_date = datetime.strptime(end_time, "%Y-%m-%d")
    
    # Initialize an empty DataFrame to store all results
    all_data = pd.DataFrame()
    
    # Loop through each month in the date range
    current_start = start_date
    while current_start < end_date:
        # Define the end of the current month
        current_end = (current_start + timedelta(days=chunk_size))
        if current_end > end_date:
            current_end = end_date
        
        # Download data for the current month
        data_chunk = download_earthquake_data_chunk(
            current_start.strftime("%Y-%m-%d"), current_end.strftime("%Y-%m-%d"), region)
        
        # If data is returned, append it to the main DataFrame
        if data_chunk:
            chunk_df = pd.read_csv(StringIO(data_chunk.decode('utf-8')))
            all_data = pd.concat([all_data, chunk_df], ignore_index=True)
        
        # Move to the next month
        current_start = current_end
    
    # Save the complete dataset to a CSV file
    all_data.to_csv(output_file, index=False)
    print(f"Data downloaded successfully and saved to {output_file}.")

In [ ]:
# Parameters for the earthquake search
start_time = "2020-01-01"    # Start date
end_time = "2024-01-01"      # End date
region = {
    "minlatitude": 24.6,     # Min latitude of the region
    "maxlatitude": 50.0,     # Max latitude
    "minlongitude": -125.0,  # Min longitude
    "maxlongitude": -65.0    # Max longitude
}
output_file = f"{data_folder}/raw/us_earthquake/us_earthquakes.csv"

# Download the earthquake data
download_earthquake_data(start_time, end_time, region, output_file, chunk_size=30)

### Loading Data

In [ ]:
df_earthquakes = pd.read_csv(f'{data_folder}/raw/us_earthquake/us_earthquakes.csv')

In [ ]:
df_earthquakes.info()

In [ ]:
pd.to_datetime(df_earthquakes.time).describe()

### Preprocessing Data

In [13]:
df_earthquakes = df_earthquakes[
    (df_earthquakes["type"] == "earthquake") & (df_earthquakes["status"] == "reviewed") 
    & (df_earthquakes['magType'] == 'ml')]
df_earthquakes = df_earthquakes.dropna(subset=['time', 'latitude', 'longitude', 'mag'])\
    .drop_duplicates(subset=['time', 'latitude', 'longitude', 'mag'], keep='first')
df_earthquakes["time"] = pd.to_datetime(df_earthquakes["time"])
df_earthquakes['coordinate'] = df_earthquakes.apply(
    lambda row: (round(row['latitude']), round(row['longitude'])), axis=1)

In [ ]:
df_earthquakes['coordinate'].value_counts()

### Getting Sequence IDs

In [15]:
df_earthquakes = df_earthquakes.sort_values(by=["coordinate", "time"]).reset_index(drop=True)

In [ ]:
seq_ids = []
seq_count = 0
last_time = df_earthquakes.loc[0, 'time']
last_coord = df_earthquakes.loc[0, 'coordinate']
max_hours = 24

for index, row in tqdm(df_earthquakes.iterrows(), total=len(df_earthquakes)):
    if row["coordinate"] != last_coord:
        seq_count += 1
    elif (row["time"] - last_time).total_seconds() / 3600 > max_hours:
        seq_count += 1
    
    seq_ids.append(seq_count)
    last_time = row["time"]
    last_coord = row["coordinate"]

df_earthquakes["seq_id"] = seq_ids

In [ ]:
df_earthquakes.head()

### Selecting Sequences

In [ ]:
df_earthquakes["seq_id"].value_counts().describe()

In [ ]:
earthquake_counts = df_earthquakes["seq_id"].value_counts()
earthquake_list = earthquake_counts[(earthquake_counts >= 5) & (earthquake_counts <= 30)].index
len(earthquake_list)

In [20]:
df_earthquakes = df_earthquakes[df_earthquakes["seq_id"].isin(earthquake_list)]

In [ ]:
df_earthquakes.groupby('seq_id')['time'].count().describe()

### Setting Event Types

In [ ]:
df_earthquakes['mag'].describe()

In [ ]:
df_earthquakes["type"] = "Small"
df_earthquakes.loc[df_earthquakes["mag"] >= 1, "type"] = "Medium"
df_earthquakes.loc[df_earthquakes["mag"] >= 2, "type"] = "Large"
df_earthquakes["type"].value_counts()

### Saving Sequences

In [ ]:
df_earthquakes.groupby('seq_id')['time'].count().describe()

In [ ]:
def get_seq_splits(df, seq_col):
    seq_ids = df[seq_col].unique().tolist()
    seq_ids_train, seq_ids_val_test = train_test_split(seq_ids, train_size=0.8, random_state=0)
    seq_ids_val, seq_ids_test = train_test_split(seq_ids_val_test, train_size=0.5, random_state=0)
    seq_splits = {seq_id: 'train' for seq_id in seq_ids_train}
    seq_splits.update({seq_id: 'dev' for seq_id in seq_ids_val})
    seq_splits.update({seq_id: 'test' for seq_id in seq_ids_test})
    print(f'train: {len(seq_ids_train)} seqs, val: {len(seq_ids_val)} seqs, test: {len(seq_ids_test)} seqs')
    return seq_splits

In [ ]:
earthquake_seq_splits = get_seq_splits(df=df_earthquakes, seq_col='seq_id')
len(earthquake_seq_splits)

In [ ]:
df_earthquakes

In [29]:
def save_seqs(
    df: pd.DataFrame, seq_col: str, seq_splits: dict,
    time_col: str, time_unit: float, type_col: str,mag_col: str,depth_col: str, seq_folder: str):
    """
    Save event sequences
    """
    dim_process = df[type_col].nunique()
    type_text2id = {type_text: type_id for type_id, type_text in enumerate(df[type_col].unique())}
    type_id2text = {type_id: type_text for type_text, type_id in type_text2id.items()}
    type_id_col = f'{type_col}_id'
    df[type_id_col] = df[type_col].map(type_text2id)
    data = {'train': [], 'dev': [], 'test': []}
    print(f'type_id2text: {type_id2text}')
    
    for seq_id, group in tqdm(df.groupby(seq_col)):
        group = group.sort_values(by=time_col).reset_index()
        split = seq_splits[seq_id]
        init_time = group[time_col].min()
        pre_event_time = init_time
        event_seq = {
            'dim_process': dim_process,
            'seq_idx': len(data[split]),
            'seq_len': len(group),
            'time_since_start': [],
            'time_since_last_event': [],
            'type_event': [],
            'type_text': [],
            'magnitude':[],
            'depth':[]
        }
        
        for index, row in group.iterrows():
            event_time = pd.to_datetime(row[time_col])
            time_since_start = (event_time - init_time).total_seconds() / time_unit
            time_since_last_event = (event_time - pre_event_time).total_seconds() / time_unit
            event_seq['time_since_start'].append(time_since_start)
            event_seq['time_since_last_event'].append(time_since_last_event)
            event_seq['type_event'].append(row[type_id_col])
            event_seq['type_text'].append(row[type_col])
            event_seq['magnitude'].append(row[mag_col])
            event_seq['depth'].append(row[depth_col])
            pre_event_time = event_time
        
        data[split].append(event_seq)

    os.makedirs(seq_folder, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        json_path = f'{seq_folder}/{split}.json'
        with open(json_path, 'w') as file:
            json.dump(data[split], file, indent=4)
        print(f'{split} saved to {json_path}')

In [ ]:
save_seqs(
    df=df_earthquakes, seq_col='seq_id', seq_splits=earthquake_seq_splits,
    time_col='time', time_unit=60*60*24, type_col='type',mag_col='mag',depth_col='depth',
    seq_folder=f'{data_folder}/us_earthquake',
)